# 03 · Common video perturbations

Apply additive noise, shot noise, flicker, motion jitter and fog, then quantify robustness (`R`) and efficiency (`LERS`).

In [ ]:
import torch
from flynet_robustness.textures import get_texture
from flynet_robustness.decoders import to_logits
from flynet_robustness.perturbations import texture_to_input_perturbed
from flynet_robustness.models import get_flynet, forward_flynet
from flynet_robustness.hex_eye import BoxEye
from flynet_robustness.decoders import load_decoder

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model1, model2, idxs1, idxs2 = get_flynet('flynet_synthesized', device=device)
decoder = load_decoder(cnn=False, cnn_id='flynet_synthesized')
renderer_hex = BoxEye(model1.connectome.config.extent, kernel_size=13)

In [ ]:
base_kwargs = dict(hit=True, radius=4.0, shape='Sphere', theta=0, time=1, fps=100,
                   stop_margin_hit=0.6, device=device, z0=15, out_size=416, return_rgb=True)
x_logits = to_logits(get_texture('Black').to(device))

def p_hit_under(perturbation, **kw):
    inp, _ = texture_to_input_perturbed(x_logits, base_kwargs, renderer_hex,
                                        cnn_id='flynet_synthesized', perturbation=perturbation, **kw)
    activity = forward_flynet(inp, model1, idxs1).flatten(end_dim=1)
    return torch.sigmoid(decoder(activity)).mean().item()

print('clean   ', p_hit_under(None))
print('AWN     ', p_hit_under('awn', sigma=0.05))
print('flicker ', p_hit_under('flicker', strength=0.05))
print('fog     ', p_hit_under('fog', contrast_scale=0.3))

## Robustness R and LERS
Sweep a perturbation magnitude, take the mean accuracy over the sweep (`R`), then normalize by log-compute (`LERS`).

In [ ]:
import numpy as np
from flynet_robustness.metrics import robustness, lers

sigmas = [0.0, 0.02, 0.04, 0.06, 0.08]
acc = [np.mean([p_hit_under('awn', sigma=s) > 0.5]) for s in sigmas]  # 1 trajectory; average more in practice
R = robustness(sigmas, acc)
print('R   =', round(R, 3))
print('LERS=', round(lers(R, 'flynet_synthesized'), 3))